# Практика · Модель у продакшені

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.md](homework.md)

У [темі 30](../30-end-to-end-project/lecture.html) ми довели детектор шахрайських
оголошень до рішення «запускати». Тут ми його **запускаємо** і дивимось, що з ним
станеться за рік.

Що зробимо:

1. навчимо модель на даних дня запуску й запамʼятаємо її якість;
2. **згенеруємо дванадцять місяців** нових оголошень, у яких повільно
   змінюється і ринок, і поведінка шахраїв;
3. поміряємо якість помісячно — і побачимо падіння;
4. порахуємо **PSI та KS** на розподілі ознаки, тобто **без жодної мітки**, і
   переконаємось, що вони кричать раніше, ніж приходить метрика якості;
5. покажемо **training/serving skew** на одній конкретній ознаці;
6. перенавчимо модель щомісяця й порівняємо з «запустили й забули».

Уся випадковість зафіксована зерном 42 — числа в тебе вийдуть точно ті самі,
що в лекції.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# зерно фіксує всю випадковість: у тебе вийдуть точно ті самі числа, що в лекції
rng = np.random.default_rng(42)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 16)
print("numpy", np.__version__, "· pandas", pd.__version__)

## 1 · Дошка, яка живе в часі

Генератор той самий, що в темі 30, з двома доробками — і саме вони роблять цю тему.

**Ринок дешевшає.** Кожен місяць типова ціна вживаного телефона падає на 4.5 %
(`PRICE_FALL = 0.955`). За рік це мінус 42 %. Це **дрейф даних**: розподіл ознаки
поїхав, а звʼязок «сильно дешевше за типову → підозріло» лишився тим самим.

**Шахраї міняють схему.** Перші чотири місяці вони роблять те саме, що й раніше:
ставлять або дуже дешеву приманку, або дуже дорогу. З пʼятого місяця частина з них
починає ставити ціну **біля типової** — і ця частина щомісяця зростає на 11 пунктів,
поки не дійде до 80 %. Це **дрейф концепції**: та сама ціна тепер означає інше.

Спрощення проти теми 30: тут немає колекційних телефонів, одруків і дублікатів —
тільки пропуски в ціні, бо саме вони впливають на модель. Тому число на старті
буде трохи інше, ніж 0.796 з теми 30.

In [ ]:
MODELS = ["Alfa A5", "Alfa A7", "Beta 12", "Beta 12 Pro", "Gamma X", "Gamma X Ultra"]
NEW_PRICE = {"Alfa A5": 5200, "Alfa A7": 7400, "Beta 12": 12000,
             "Beta 12 Pro": 17500, "Gamma X": 24000, "Gamma X Ultra": 34000}
MODEL_SHARE = [0.24, 0.22, 0.18, 0.16, 0.12, 0.08]
CONDITION_K = {"нове": 1.0, "дуже добре": 0.88, "добре": 0.75, "задовільне": 0.58}
MEMORY_K = {64: 0.85, 128: 1.0, 256: 1.15, 512: 1.32}

PRICE_FALL = 0.955   # ринок щомісяця дешевшає — дрейф даних
NEAR_STEP = 0.11     # приріст частки шахраїв «біля типової ціни» — дрейф концепції


def make_month(n_ads, month, rng):
    '''Оголошення одного місяця. month=0 — день запуску, далі — життя в бою.'''
    model = rng.choice(MODELS, size=n_ads, p=MODEL_SHARE)
    year = rng.integers(2017, 2025, size=n_ads)
    condition = rng.choice(list(CONDITION_K), size=n_ads, p=[0.08, 0.32, 0.42, 0.18])
    memory_gb = rng.choice(list(MEMORY_K), size=n_ads, p=[0.30, 0.38, 0.24, 0.08])
    account_age = np.round(rng.exponential(420, size=n_ads) + 3).astype(int)

    base = np.array([NEW_PRICE[m] for m in model])
    wear = 0.82 ** (2024 - year)
    k_condition = np.array([CONDITION_K[c] for c in condition])
    k_memory = np.array([MEMORY_K[m] for m in memory_gb])
    # ключовий рядок теми: та сама модель того самого року з місяцями коштує дешевше
    typical_price = base * wear * k_condition * k_memory * (PRICE_FALL ** month)
    price = typical_price * rng.lognormal(0, 0.13, size=n_ads)

    is_fraud = rng.random(n_ads) < (0.10 + 0.30 * np.exp(-account_age / 120))

    # з пʼятого місяця дедалі більше шахраїв ставлять ціну біля типової
    near_share = min(0.80, NEAR_STEP * max(0, month - 4))
    draw = rng.random(n_ads)
    near = is_fraud & (draw < near_share)
    old_school = is_fraud & ~near
    cheap_bait = old_school & (rng.random(n_ads) < 0.74)
    pricey_bait = old_school & ~cheap_bait

    price[near] = typical_price[near] * rng.uniform(0.85, 1.10, near.sum())
    price[cheap_bait] = typical_price[cheap_bait] * rng.uniform(0.20, 0.45, cheap_bait.sum())
    price[pricey_bait] = typical_price[pricey_bait] * rng.uniform(2.6, 3.8, pricey_bait.sum())
    price = np.round(price, -1)

    # шахрай частіше ховає ціну, ніж чесний продавець — це теж сигнал
    hidden = rng.random(n_ads) < np.where(is_fraud, 0.25, 0.03)
    price[hidden] = np.nan

    return pd.DataFrame({"model": model, "year": year, "condition": condition,
                         "memory_gb": memory_gb, "account_age": account_age,
                         "price": price, "is_fraud": is_fraud.astype(int),
                         "month": month})


launch_board = make_month(2000, 0, rng)
board_by_month = {m: make_month(1500, m, rng) for m in range(1, 13)}

print("день запуску:", launch_board.shape,
      "· шахрайських", f"{launch_board['is_fraud'].mean() * 100:.1f} %")
print("кожен наступний місяць: 1 500 оголошень, усього 12 місяців")

Перше, що варто перевірити, — **чи змінилась частка шахрайських оголошень**. Це третій
вид дрейфу, дрейф міток, і його плутають із двома іншими найчастіше.

In [ ]:
fraud_share = pd.DataFrame({
    "місяць": list(range(1, 13)),
    "шахрайських, %": [round(board_by_month[m]["is_fraud"].mean() * 100, 1) for m in range(1, 13)],
    "медіанна ціна, грн": [int(board_by_month[m]["price"].median()) for m in range(1, 13)],
})
print(fraud_share.to_string(index=False))
print()
first, last = fraud_share["медіанна ціна, грн"].iloc[[0, -1]]
print("Частка шахрайських тримається близько 16 % — дрейфу МІТОК немає.")
print(f"А медіанна ціна впала з {first} до {last} грн, тобто на "
      f"{(1 - last / first) * 100:.0f} % — це дрейф ДАНИХ.")

## 2 · Модель на день запуску

Усе як у темі 30: відкладаємо чверть оголошень, будуємо карту типових цін
**тільки на навчальній частині**, робимо з неї ознаку `price_deviation` —
модуль логарифма відношення ціни до типової, — і навчаємо логістичну регресію
всередині конвеєра.

`price_map` — це і є та сама «медіанна ціна моделі», навколо якої обертається
вся тема. Запамʼятай момент: карту пораховано **сьогодні** й вона поїде разом
із моделлю в бій.

In [ ]:
train_raw, test_raw = train_test_split(
    launch_board, test_size=0.25, random_state=42, stratify=launch_board["is_fraud"])

# карта типових цін: медіана всередині групи «модель + рік», лише по навчальній частині
price_map = train_raw.groupby(["model", "year"])["price"].median()
overall_median_price = train_raw["price"].median()


def add_features(board, reference_map, fallback_price):
    '''Дві ознаки з ціни. reference_map — той самий довідник типових цін;
    саме він і буде героєм розділу про training/serving skew.'''
    out = board.copy()
    keys = pd.MultiIndex.from_arrays([out["model"], out["year"]])
    # групам, яких у довіднику немає, підставляємо загальну медіану
    out["typical_price"] = keys.map(reference_map).fillna(fallback_price)
    # модуль логарифма робить «утричі дешевше» й «утричі дорожче» однаково великими
    out["price_deviation"] = np.abs(np.log(out["price"] / out["typical_price"]))
    out["price_known"] = out["price"].notna().astype(int)
    return out


NUMERIC = ["year", "memory_gb", "account_age", "price_deviation", "price_known"]
CATEGORICAL = ["model", "condition"]


def build_pipeline():
    '''Свіжий необучений конвеєр. Потрібен окремою функцією, бо далі
    ми навчатимемо його заново дванадцять разів.'''
    numeric_path = Pipeline([("impute", SimpleImputer(strategy="median")),
                             ("scale", StandardScaler())])
    categorical_path = OneHotEncoder(handle_unknown="ignore")
    prep = ColumnTransformer([("numeric", numeric_path, NUMERIC),
                              ("categorical", categorical_path, CATEGORICAL)])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000, C=10))])


train_features = add_features(train_raw, price_map, overall_median_price)
test_features = add_features(test_raw, price_map, overall_median_price)

launch_model = build_pipeline()
launch_model.fit(train_features[NUMERIC + CATEGORICAL], train_features["is_fraud"])

test_pred = launch_model.predict(test_features[NUMERIC + CATEGORICAL])
launch_f1 = f1_score(test_features["is_fraud"], test_pred)
print("=== ЯКІСТЬ НА ДЕНЬ ЗАПУСКУ (500 відкладених оголошень) ===")
print("F1        :", round(launch_f1, 3))
print("precision :", round(precision_score(test_features["is_fraud"], test_pred), 3))
print("recall    :", round(recall_score(test_features["is_fraud"], test_pred), 3))
print()
print("Саме це число ми покажемо керівництву. Подивимось, скільки воно проживе.")

### Перевірка: F1 руками проти бібліотечного

Перш ніж будувати на цьому числі цілу тему, переконаймось, що всередині `f1_score`
немає магії — воно дістається з чотирьох клітинок матриці помилок.

In [ ]:
honest_ok, false_alarms, missed, caught = confusion_matrix(
    test_features["is_fraud"], test_pred).ravel()

our_precision = caught / (caught + false_alarms)
our_recall = caught / (caught + missed)
our_f1 = 2 * our_precision * our_recall / (our_precision + our_recall)

print("наш F1     :", round(our_f1, 6))
print("f1_score   :", round(launch_f1, 6))
assert np.allclose(our_f1, launch_f1), "розрахунок розійшовся!"
print("✅ збігається")

## 3 · Дванадцять місяців у бою

Модель у сервісі **заморожена**: і ваги, і довідник `price_map` — це один
артефакт, який ніхто не чіпає. Кожен місяць приходить нова тисяча пʼятсот
оголошень, і ми міряємо якість на них.

Важливо: у справжньому бою цих чисел ти **не бачиш** — щоб їх порахувати,
потрібні правильні відповіді, а вони приходять пізніше. Тут ми їх знаємо,
бо самі згенерували, і це дає нам рідкісну змогу побачити правду.

In [ ]:
monthly = []
for month in range(1, 13):
    board = board_by_month[month]
    scored = add_features(board, price_map, overall_median_price)   # довідник із дня запуску
    prediction = launch_model.predict(scored[NUMERIC + CATEGORICAL])
    monthly.append({
        "місяць": month,
        "F1": round(f1_score(board["is_fraud"], prediction), 3),
        "precision": round(precision_score(board["is_fraud"], prediction, zero_division=0), 3),
        "recall": round(recall_score(board["is_fraud"], prediction), 3),
        "частка тривог": round(float(prediction.mean()), 3),
        "мед. відхилення": round(float(scored["price_deviation"].median()), 3),
    })

monthly_table = pd.DataFrame(monthly)
print(monthly_table.to_string(index=False))

Читаємо не колонку F1, а те, що з нею сталось. Перші чотири місяці якість тримається
біля стартової. З пʼятого починає осідати, а до дванадцятого від неї лишається
приблизно третина.

Зверни увагу на дві останні колонки: **частка тривог зросла вдвічі** (0.161 → 0.333),
а медіанне відхилення ціни від «типової» — у два з половиною рази. Обидва числа
рахуються **без жодної мітки**. Це і є те, що видно в бою.

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 3.6))
ax.plot(monthly_table["місяць"], monthly_table["F1"], marker="o", color="#c2185b", label="F1 у бою")
ax.axhline(launch_f1, ls="--", lw=1.2, color="#0f766e", label=f"обіцянка з тесту: {launch_f1:.3f}")
ax.axhline(0.80, ls=":", lw=1.2, color="#c2620f", label="домовлена межа 0.80")
ax.set_xlabel("місяців від запуску")
ax.set_ylabel("F1")
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.set_title("Модель, яку запустили й забули")
plt.tight_layout()
plt.show()

below = monthly_table[monthly_table["F1"] < 0.80]["місяць"]
print("нижче межі 0.80 модель опустилась у місяці:", int(below.iloc[0]))

## 4 · А тепер без міток

Тепер найважливіше в темі. У бою модератор підтверджує шахрайство не одразу:
скарга приходить, коли покупець уже втратив передоплату. Візьмімо **затримку
у два місяці** — це оптимістично.

Отже, у місяці 7 ти маєш метрику якості лише за місяць 5. А от **розподіл ознак**
доступний одразу: щоб порахувати, як виглядає `price_deviation` у сьогоднішніх
оголошеннях, правильні відповіді не потрібні взагалі.

Двома найпоширенішими мірами такого зсуву є **PSI** та **KS**.

**PSI** (population stability index, індекс стабільності популяції) робить просту річ.
Розділи стару вибірку на десять кошиків так, щоб у кожен потрапило по 10 % значень.
Порахуй, яка частка **нових** значень падає в кожен кошик. Якщо розподіл не змінився,
частки будуть по 10 %. PSI підсумовує по всіх кошиках різницю часток, помножену на
логарифм їхнього відношення:

$$\mathrm{PSI} = \sum_{i=1}^{10} (n_i - s_i)\,\ln\frac{n_i}{s_i}$$

Тут `s` — частка старих значень у кошику, `n` — частка нових, а `ln` — натуральний
логарифм, той самий, що й у нашій ознаці. Множник `(n − s)` каже, **наскільки**
кошик змінився, логарифм — **у скільки разів**. Обидві частки маленькі, тому число
виходить маленьким: усталені пороги — 0.1 (варто подивитись) і 0.25 (зсув значний).

**KS** (статистика Колмогорова — Смирнова) міряє інше: максимальний розрив між двома
накопиченими частками. Простими словами — стань у найгіршій точці шкали й спитай,
яка частка старих значень лежить лівіше і яка частка нових. Найбільша різниця цих
двох чисел і є KS. Воно завжди між 0 (розподіли збігаються) і 1 (не перетинаються).

In [ ]:
reference_deviation = train_features["price_deviation"].dropna().to_numpy()

# межі кошиків — по децилях СТАРОЇ вибірки, тому в кожному рівно 10 % старих значень
bin_edges = np.quantile(reference_deviation, np.linspace(0, 1, 11))
bin_edges[0], bin_edges[-1] = -np.inf, np.inf


def bucket_share(values):
    '''Яка частка значень падає в кожен із десяти кошиків.'''
    clean = values[~np.isnan(values)]
    counts = np.histogram(clean, bins=bin_edges)[0]
    # порожній кошик дав би ділення на нуль у логарифмі, тому підстраховуємось
    return np.clip(counts / len(clean), 1e-4, None)


reference_share = bucket_share(reference_deviation)


def psi(values):
    '''Population stability index: наскільки нові значення розповзлись по кошиках.'''
    new_share = bucket_share(values)
    return float(np.sum((new_share - reference_share) * np.log(new_share / reference_share)))


def ks(values):
    '''Максимальний розрив між двома накопиченими частками.'''
    new = np.sort(values[~np.isnan(values)])
    old = np.sort(reference_deviation)
    grid = np.concatenate([old, new])
    old_cdf = np.searchsorted(old, grid, side="right") / len(old)
    new_cdf = np.searchsorted(new, grid, side="right") / len(new)
    return float(np.max(np.abs(old_cdf - new_cdf)))


print("PSI старої вибірки самої з собою:", round(psi(reference_deviation), 6))
print("KS  старої вибірки самої з собою:", round(ks(reference_deviation), 6))
print()
print("Нуль і нуль — рівно те, чого ми чекали: розподіл не змінився ні на йоту.")

### Перевірка: наш KS проти бібліотечного

`scipy` приїжджає разом зі `scikit-learn`, тому порівняти є з чим.

In [ ]:
from scipy.stats import ks_2samp

sample_month = add_features(board_by_month[8], price_map, overall_median_price)
sample_values = sample_month["price_deviation"].to_numpy()

our_ks = ks(sample_values)
library_ks = ks_2samp(reference_deviation, sample_values[~np.isnan(sample_values)]).statistic

print("наш KS      :", round(our_ks, 6))
print("ks_2samp    :", round(library_ks, 6))
assert np.allclose(our_ks, library_ks), "розрахунок розійшовся!"
print("✅ збігається")

In [ ]:
LABEL_LAG = 2   # мітки за місяць m стають відомі аж у місяці m + 2

drift_rows = []
for month in range(1, 13):
    scored = add_features(board_by_month[month], price_map, overall_median_price)
    values = scored["price_deviation"].to_numpy()
    known_month = month - LABEL_LAG
    drift_rows.append({
        "місяць": month,
        "PSI": round(psi(values), 3),
        "KS": round(ks(values), 3),
        "частка тривог": monthly_table.loc[month - 1, "частка тривог"],
        "F1 насправді": monthly_table.loc[month - 1, "F1"],
        "F1, який ти бачиш": (monthly_table.loc[known_month - 1, "F1"]
                              if known_month >= 1 else np.nan),
    })

drift_table = pd.DataFrame(drift_rows)
print(drift_table.to_string(index=False))

Ось воно, головне число теми.

* PSI переходить позначку **0.1** у місяці 5 — тоді ж, коли якість уперше падає
  нижче домовлених 0.80. Але про падіння якості ти в цю мить **не знаєш**: у руках
  у тебе метрика за місяць 3, і вона показує 0.879.
* PSI переходить **0.25** у місяці 7. Метрика якості, яку ти бачиш у місяці 7, —
  це якість місяця 5.
* Реальний провал (F1 = 0.777, місяць 5) доходить до тебе **у місяці 7**.

Два місяці — це рівно затримка міток. Увесь цей час детектор працює гірше, ніж
обіцяв, а звіт показує зелене.

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 3.8))
ax.plot(drift_table["місяць"], drift_table["F1 насправді"], marker="o",
        color="#c2185b", label="F1 насправді")
ax.plot(drift_table["місяць"], drift_table["F1, який ти бачиш"], marker="s", ls="--",
        color="#0f766e", label=f"F1, який ти бачиш (мітки на {LABEL_LAG} міс. пізніше)")
ax.plot(drift_table["місяць"], drift_table["PSI"].clip(upper=1.0), marker="^",
        color="#c2620f", label="PSI (обрізано на 1.0)")
ax.axhline(0.25, ls=":", lw=1.2, color="#c2620f")
ax.set_xlabel("місяців від запуску")
ax.set_ylabel("значення")
ax.legend(fontsize=9)
ax.set_title("Сигнал без міток приходить раніше за метрику якості")
plt.tight_layout()
plt.show()

print("PSI перетнув 0.10 у місяці:",
      int(drift_table.loc[drift_table["PSI"] > 0.10, "місяць"].iloc[0]))
print("PSI перетнув 0.25 у місяці:",
      int(drift_table.loc[drift_table["PSI"] > 0.25, "місяць"].iloc[0]))

## 5 · Training/serving skew на одній ознаці

Тепер найдорожча пастка теми, і вона не про дрейф. Ознака `price_deviation`
рахується з довідника типових цін. Питання одне: **з якого саме**.

* **Як рахували в зошиті.** Ти сидиш із таблицею за дванадцять місяців і міряєш
  якість помісячно. Довідник ти щоразу будуєш із того самого місяця, який зараз
  оцінюєш, — бо він у тебе в руках, і це виглядає природно.
* **Як доступно в бою.** У мить, коли оголошення публікують, сервіс має рівно один
  довідник — той, що поїхав разом із моделлю в день запуску.

Той самий код, та сама формула, різні числа. Подивимось на конкретне оголошення.

In [ ]:
month_10 = board_by_month[10]
fresh_map_10 = month_10.groupby(["model", "year"])["price"].median()

as_in_notebook = add_features(month_10, fresh_map_10, month_10["price"].median())
as_in_production = add_features(month_10, price_map, overall_median_price)

# беремо перше-ліпше чесне оголошення популярної моделі з відомою ціною
sample = as_in_production[(as_in_production["is_fraud"] == 0)
                          & (as_in_production["model"] == "Beta 12")
                          & (as_in_production["year"] == 2021)
                          & as_in_production["price"].notna()].index[0]

print("оголошення: Beta 12, 2021 рік, ціна", int(month_10.loc[sample, "price"]), "грн")
print()
print("як рахували в зошиті : типова", int(as_in_notebook.loc[sample, "typical_price"]),
      "грн → відхилення", round(as_in_notebook.loc[sample, "price_deviation"], 3))
print("як доступно в бою    : типова", int(as_in_production.loc[sample, "typical_price"]),
      "грн → відхилення", round(as_in_production.loc[sample, "price_deviation"], 3))
print()
print("Одне оголошення, одна формула — і відхилення відрізняється втричі.")

Тепер те саме на всіх дванадцяти місяцях разом. Порівняємо три способи будувати
довідник:

1. **зошит** — карта з того самого місяця, який оцінюємо (у бою так не можна:
   у мить публікації оголошень цього місяця ще немає);
2. **бій, заморожено** — карта з дня запуску, тобто те, що реально їде в сервіс;
3. **бій, оновлюване** — карта з **минулого** місяця. Це чесно доступно в бою,
   і саме так треба було зробити з самого початку.

In [ ]:
truth, pred_notebook, pred_frozen, pred_rolling = [], [], [], []
for month in range(1, 13):
    board = board_by_month[month]
    truth.append(board["is_fraud"].to_numpy())

    fresh_map = board.groupby(["model", "year"])["price"].median()
    pred_notebook.append(launch_model.predict(
        add_features(board, fresh_map, board["price"].median())[NUMERIC + CATEGORICAL]))

    pred_frozen.append(launch_model.predict(
        add_features(board, price_map, overall_median_price)[NUMERIC + CATEGORICAL]))

    previous = board_by_month[month - 1] if month > 1 else train_raw
    rolling_map = previous.groupby(["model", "year"])["price"].median()
    pred_rolling.append(launch_model.predict(
        add_features(board, rolling_map, previous["price"].median())[NUMERIC + CATEGORICAL]))

truth = np.concatenate(truth)
f1_notebook = f1_score(truth, np.concatenate(pred_notebook))
f1_frozen = f1_score(truth, np.concatenate(pred_frozen))
f1_rolling = f1_score(truth, np.concatenate(pred_rolling))

print("F1 на всіх 18 000 оголошеннях року:")
print("  зошит (свіжа карта)     :", round(f1_notebook, 3))
print("  бій, заморожена карта   :", round(f1_frozen, 3))
print("  бій, оновлювана карта   :", round(f1_rolling, 3))
print()
print("розрив зошит ↔ бій:", round(f1_notebook - f1_frozen, 3))
print()
print("Оновлювана карта дає 0.730 — тобто зошит не брехав про досяжну якість.")
print("Він брехав про ТУ САМУ систему, яку ми запустили.")

Розрив 0.111 — це не помилка вимірювання й не невдача моделі. Це різниця між
двома способами порахувати одну ознаку.

І запобіжник тут не технічний, а дисциплінарний: **ознаку рахує один шматок коду,
і в зошиті, і в сервісі**. Щойно та сама формула існує у двох місцях, вони починають
розходитись — не сьогодні, то через три місяці, коли хтось поправить одну з них.

## 6 · Перенавчання

Модель постаріла — навчимо нову. Але тут є пастка, про яку легко забути: перенавчати
можна лише на **розмічених** даних, а мітки запізнюються на два місяці. Отже, у
місяці 7 найсвіжіші правильні відповіді, які в нас є, — за місяць 5.

Стратегія: щомісяця беремо три останні **розмічені** місяці, навчаємо модель заново,
а довідник типових цін будуємо з минулого місяця — його мітки не потрібні.

In [ ]:
retrain_rows = []
for month in range(1, 13):
    board = board_by_month[month]
    last_labelled = month - LABEL_LAG

    if last_labelled >= 1:
        history = pd.concat([board_by_month[k]
                             for k in range(max(1, last_labelled - 2), last_labelled + 1)],
                            ignore_index=True)
        trained_on = f"{max(1, last_labelled - 2)}–{last_labelled}"
    else:
        history = train_raw                      # міток ще немає, живемо на стартових
        trained_on = "старт"

    history_map = history.groupby(["model", "year"])["price"].median()
    history_features = add_features(history, history_map, history["price"].median())
    monthly_model = build_pipeline()
    monthly_model.fit(history_features[NUMERIC + CATEGORICAL], history_features["is_fraud"])

    # у бою довідник будуємо з минулого місяця: міток він не потребує
    previous = board_by_month[month - 1] if month > 1 else train_raw
    serving_map = previous.groupby(["model", "year"])["price"].median()
    scored = add_features(board, serving_map, previous["price"].median())
    prediction = monthly_model.predict(scored[NUMERIC + CATEGORICAL])

    retrain_rows.append({
        "місяць": month,
        "навчено на місяцях": trained_on,
        "F1 без перенавчання": monthly_table.loc[month - 1, "F1"],
        "F1 з перенавчанням": round(f1_score(board["is_fraud"], prediction), 3),
    })

retrain_table = pd.DataFrame(retrain_rows)
print(retrain_table.to_string(index=False))
print()
late = retrain_table[retrain_table["місяць"] >= 5]
print("середній F1 за місяці 5–12:")
print("  запустили й забули :", round(late["F1 без перенавчання"].mean(), 3))
print("  перенавчаємо щомісяця:", round(late["F1 з перенавчанням"].mean(), 3))

Два висновки, і другий важливіший.

**Перший:** перенавчання працює. Середній F1 за другу половину року — 0.647 замість
0.517. Це різниця між «детектор гальмує» і «детектор здався».

**Другий:** перенавчання **не рятує**. До дванадцятого місяця навіть свіжа модель
дає близько 0.5. Причина не в алгоритмі: модель, навчена на місяцях 8–10, ловить
шахраїв, які працювали за схемою місяців 8–10, а в місяці 12 схема вже інша.
Поки мітки їдуть два місяці, модель приречена бігти позаду.

Що з цим роблять насправді: скорочують затримку міток. Кнопка «поскаржитись» на
оголошенні дає розмітку за години, а не за місяці, — і це дешевша інвестиція,
ніж будь-яка нова модель.

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 3.6))
ax.plot(retrain_table["місяць"], retrain_table["F1 без перенавчання"], marker="o",
        color="#c2185b", label="запустили й забули")
ax.plot(retrain_table["місяць"], retrain_table["F1 з перенавчанням"], marker="s",
        color="#0f766e", label="перенавчання щомісяця")
ax.axhline(0.80, ls=":", lw=1.2, color="#c2620f", label="домовлена межа 0.80")
ax.set_xlabel("місяців від запуску")
ax.set_ylabel("F1")
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.set_title("Перенавчання допомагає, але не наздоганяє")
plt.tight_layout()
plt.show()
print("Обидві криві йдуть униз. Верхня — повільніше.")

---

## Завдання

### 🟢 Рівень 1 — база

Пороги для PSI (0.1 і 0.25) ми взяли як усталені, але для конкретної системи їх
підбирають. Побудуй таблицю: для кожного місяця — PSI, KS і `частка тривог`. Знайди,
у якому місяці кожен із трьох сигналів **уперше** перетинає межу: PSI > 0.1,
KS > 0.15, частка тривог більша за стартову в 1.2 раза.

*Зроблено, якщо* названо три місяці й одним реченням сказано, який із трьох сигналів
спрацював би першим і чи не подав би він при цьому фальшивої тривоги.

### 🟡 Рівень 2 — плюс

Затримка міток `LABEL_LAG = 2` — це припущення. Порахуй, як змінюється **дата,
коли ти дізнаєшся про поломку**, для затримок 0, 1, 2, 4 місяці. Поломкою вважай
перший місяць, у якому справжній F1 упав нижче 0.80.

*Зроблено, якщо* є таблиця «затримка → місяць, у якому дізнався» і висновок, скільки
місяців роботи гіршого детектора коштує кожен місяць затримки розмітки.

### 🔴 Рівень 3 — виклик

Зроби **перенавчання за тригером** замість перенавчання за розкладом. Правило:
модель перенавчається лише тоді, коли PSI за поточний місяць перевищив 0.25 (і не
частіше, ніж раз на три місяці). Порівняй із щомісячним перенавчанням за двома
числами: середній F1 за місяці 5–12 і **кількість перенавчань** за рік.

*Зроблено, якщо* є обидва числа для обох стратегій і чесна відповідь, яка з них
краща для дошки оголошень — з урахуванням того, що кожне перенавчання коштує
робочого часу людини й несе ризик випустити гіршу модель.

### Підказки

* Місяць першого перетину зручно шукати через `таблиця[умова]["місяць"].iloc[0]`.
* Для рівня 3 достатньо прапорця `months_since_retrain` і копії циклу з розділу 6.
* Якщо число не збіглося з лекцією — перевір, чи не змістився порядок викликів `rng`:
  генератор віддає числа послідовно, і зайвий виклик зсуває всю решту.